In [1]:
import os
import zipfile
import shutil
import pandas as pd


def get_base_dir() -> str:
    """获取当前 Notebook 的工作目录（即本文件所在的文件夹）。"""
    return os.getcwd()


def extract_date_token(source_name: str):
    """从文件名中提取形如 20160104 的 8 位日期字符串；若不存在则返回 None。"""
    base = os.path.basename(source_name)
    for token in base.split("."):
        if len(token) == 8 and token.isdigit():
            return token
    return None


def get_output_dir_for_source(output_root_base: str, source_name: str) -> str:
    """根据文件名中的日期生成类似 20160104stocks_output/source_name 的输出目录。"""
    date_token = extract_date_token(source_name)
    if date_token:
        root = os.path.join(output_root_base, f"{date_token}stocks_output")
    else:
        root = os.path.join(output_root_base, "stocks_output")
    return os.path.join(root, source_name)


def has_output_for_source(output_root_base: str, source_name: str) -> bool:
    """判断某个原始文件是否已经拆分过（对应输出目录中是否已有 csv 文件）。"""
    folder = get_output_dir_for_source(output_root_base, source_name)
    if not os.path.exists(folder):
        return False
    for fn in os.listdir(folder):
        if fn.lower().endswith(".csv"):
            return True
    return False


def read_csv_flexible(path: str):
    """尝试用常见编码读取 CSV 文件，失败则返回 None。"""
    for enc in ("utf-8", "gbk"):
        try:
            return pd.read_csv(path, sep=",", header=None, dtype=str, encoding=enc)
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"读取文件失败，已跳过: {path}, 错误: {e}")
            return None
    print(f"无法用常见编码读取文件，已跳过: {path}")
    return None


def process_data_file_with_pandas(
    data_path: str,
    source_name: str,
    output_root_base: str,
    stock_code_column: int = 5,
):
    """使用 pandas 读取单个数据文件，按股票代码拆分并输出 csv 文件。

    返回该文件中包含的股票代码集合。
    """
    print(f"读取数据文件: {data_path}")
    df = read_csv_flexible(data_path)
    if df is None:
        return set()

    if stock_code_column >= df.shape[1]:
        print(
            f"{source_name} 不存在第 {stock_code_column + 1} 列，"
            "无法识别股票代码，已跳过。"
        )
        return set()

    code_series = df.iloc[:, stock_code_column].dropna()
    codes = code_series.unique()
    print(f"{source_name} 中包含 {len(codes)} 只股票。")

    output_dir = get_output_dir_for_source(output_root_base, source_name)
    os.makedirs(output_dir, exist_ok=True)

    all_codes = set()

    for code in codes:
        all_codes.add(code)
        # 修复：直接用 df 列构造 mask，避免 code_series.dropna() 导致索引不对齐
        col = df.iloc[:, stock_code_column]
        mask = col.notna() & (col.astype(str).str.strip() == str(code).strip())
        stock_df = df[mask]
        final_path = os.path.join(output_dir, f"stock_{code}.csv")
        stock_df.to_csv(final_path, index=False, header=False)
        print(f"股票代码 {code} 已保存为: {final_path}")

    return all_codes


# ==== 主执行逻辑（在 Notebook 中一键运行这个单元即可） ====
base_dir = get_base_dir()
ZIP_FOLDER = "zip_files"  # 压缩文件所在文件夹（所有 .zip 请放入此文件夹）
zip_scan_root = os.path.join(base_dir, ZIP_FOLDER)
output_root_base = base_dir  # 按日期在当前目录下生成类似 20160104stocks_output 的文件夹
temp_root = os.path.join(base_dir, "_temp_unzipped")

print(f"当前工作目录: {base_dir}")
print(f"压缩文件扫描目录: {zip_scan_root}")

# 清理临时解压目录
shutil.rmtree(temp_root, ignore_errors=True)
os.makedirs(temp_root, exist_ok=True)

all_stock_codes = set()

# 1. 先处理 zip_files 文件夹内所有 zip 文件
print("\n开始处理压缩文件 (.zip)...")
if os.path.isdir(zip_scan_root):
    for root, _, files in os.walk(zip_scan_root):
        for file in files:
            if not file.lower().endswith(".zip"):
                continue
            zip_path = os.path.join(root, file)
            print(f"\n处理压缩文件: {zip_path}")
            with zipfile.ZipFile(zip_path, "r") as zf:
                for member in zf.infolist():
                    if member.is_dir():
                        continue
                    inner_name = os.path.basename(member.filename)
                    if not inner_name:
                        continue
                    source_name = inner_name
                    if has_output_for_source(output_root_base, source_name):
                        print(f"{source_name} 已经有拆分结果，跳过。")
                        continue
                    temp_path = os.path.join(temp_root, f"{os.path.basename(zip_path)}__{inner_name}")
                    os.makedirs(os.path.dirname(temp_path), exist_ok=True)
                    with zf.open(member) as src, open(temp_path, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    codes = process_data_file_with_pandas(
                        data_path=temp_path,
                        source_name=source_name,
                        output_root_base=output_root_base,
                        stock_code_column=5,
                    )
                    all_stock_codes.update(codes)
                    os.remove(temp_path)
else:
    print(f"未找到文件夹 {zip_scan_root}，请将压缩文件放入该文件夹。")

# 2. 再处理 zip_files 内未压缩的数据文件（非 zip、非 .py/.ipynb），同样按文件名拆分保存
print("\n开始处理 zip_files 内未压缩数据文件...")
if os.path.isdir(zip_scan_root):
    for root, _, files in os.walk(zip_scan_root):
        for file in files:
            lower = file.lower()
            if lower.endswith(".zip") or lower.endswith(".py") or lower.endswith(".ipynb"):
                continue

            data_path = os.path.join(root, file)
            source_name = os.path.basename(data_path)

            if has_output_for_source(output_root_base, source_name):
                print(f"{source_name} 已经处理过，跳过。")
                continue

            codes = process_data_file_with_pandas(
                data_path=data_path,
                source_name=source_name,
                output_root_base=output_root_base,
                stock_code_column=5,
            )
            all_stock_codes.update(codes)
else:
    print(f"未找到文件夹 {zip_scan_root}，跳过未压缩数据文件处理。")

# 清理临时解压目录（不保留解压后的原始文件）
shutil.rmtree(temp_root, ignore_errors=True)

print(f"\n全部处理完成，合计不同股票数量: {len(all_stock_codes)}")
print("拆分后的 CSV 文件按原始文件名分别存放在各自的日期目录中，例如 20160104stocks_output/原始文件名/stock_代码.csv")


当前工作目录: /Users/yuiqin/Desktop/股票dataset
压缩文件扫描目录: /Users/yuiqin/Desktop/股票dataset/zip_files

开始处理压缩文件 (.zip)...
未找到文件夹 /Users/yuiqin/Desktop/股票dataset/zip_files，请将压缩文件放入该文件夹。

开始处理 zip_files 内未压缩数据文件...
未找到文件夹 /Users/yuiqin/Desktop/股票dataset/zip_files，跳过未压缩数据文件处理。

全部处理完成，合计不同股票数量: 0
拆分后的 CSV 文件按原始文件名分别存放在各自的日期目录中，例如 20160104stocks_output/原始文件名/stock_代码.csv


In [8]:
# ========== 统计每个 stocks_output + 提取每日期 15 只共同股票到同一文件夹 ==========
import os
import re
import shutil

BASE = os.getcwd()
SKIP_CSV_COUNT_FOR = "20160104stocks_output"  # 此文件夹只统计子文件夹，不统计 CSV 数量
COMMON_STOCKS_DIR = "15_common_stocks"
NUM_COMMON = 15


def _stock_code_from_filename(fname: str) -> str:
    """从 stock_XXX.csv 或 stock_       1301 .csv 中提取并规范化股票代码。"""
    if not fname.startswith("stock_") or not fname.endswith(".csv"):
        return ""
    return fname[6:-4].strip()


def _get_stock_codes_from_dir(path: str) -> set:
    """从目录中所有 stock_*.csv 文件名提取股票代码集合。"""
    codes = set()
    if not os.path.isdir(path):
        return codes
    for f in os.listdir(path):
        if f.startswith("stock_") and f.endswith(".csv"):
            c = _stock_code_from_filename(f)
            if c:
                codes.add(c)
    return codes


def _count_stock_csv_in_dir(path: str) -> int:
    """统计目录下 stock_*.csv 文件个数。"""
    if not os.path.isdir(path):
        return 0
    return sum(1 for f in os.listdir(path) if f.startswith("stock_") and f.endswith(".csv"))


# 查找所有 *stocks_output 目录（仅匹配 8 位数字+stocks_output）
stocks_output_dirs = []
for name in os.listdir(BASE):
    if re.match(r"^\d{8}stocks_output$", name):
        full = os.path.join(BASE, name)
        if os.path.isdir(full):
            stocks_output_dirs.append(name)
stocks_output_dirs.sort()

print("=" * 60)
print("各 stocks_output 统计（每个目录下两个子文件夹及股票 CSV 数量）")
print("=" * 60)

stats_rows = []

for dirname in stocks_output_dirs:
    dirpath = os.path.join(BASE, dirname)
    # 只考虑“子文件夹”（目录），排除 15_common_stocks 及根目录下的散落 csv
    subdirs = [
        d for d in os.listdir(dirpath)
        if os.path.isdir(os.path.join(dirpath, d)) and d != COMMON_STOCKS_DIR
    ]
    subdirs.sort()

    if len(subdirs) < 2:
        print(f"\n【{dirname}】 子文件夹数量: {len(subdirs)}（不足 2 个，跳过共同股票提取）")
        c1 = _count_stock_csv_in_dir(os.path.join(dirpath, subdirs[0])) if subdirs else 0
        c2 = _count_stock_csv_in_dir(os.path.join(dirpath, subdirs[1])) if len(subdirs) > 1 else 0
        stats_rows.append((dirname, subdirs, c1 if dirname != SKIP_CSV_COUNT_FOR else "(不统计)", c2 if dirname != SKIP_CSV_COUNT_FOR else "(不统计)"))
        continue

    folder1, folder2 = subdirs[0], subdirs[1]
    path1 = os.path.join(dirpath, folder1)
    path2 = os.path.join(dirpath, folder2)

    skip_csv_count = dirname == SKIP_CSV_COUNT_FOR
    if skip_csv_count:
        cnt1 = cnt2 = "(不统计)"
    else:
        cnt1 = _count_stock_csv_in_dir(path1)
        cnt2 = _count_stock_csv_in_dir(path2)

    print(f"\n【{dirname}】")
    print(f"  子文件夹1: {folder1}  →  股票 CSV 数: {cnt1}")
    print(f"  子文件夹2: {folder2}  →  股票 CSV 数: {cnt2}")
    stats_rows.append((dirname, subdirs, cnt1, cnt2))

    # 共同股票：两文件夹都有的优先；不足 15 只时从两文件夹并集中补足
    codes1 = _get_stock_codes_from_dir(path1)
    codes2 = _get_stock_codes_from_dir(path2)
    common = codes1 & codes2
    selected = sorted(common)[:NUM_COMMON]
    if len(selected) < NUM_COMMON:
        rest = sorted((codes1 | codes2) - set(selected))
        for code in rest:
            if len(selected) >= NUM_COMMON:
                break
            selected.append(code)

    out_subdir = os.path.join(dirpath, COMMON_STOCKS_DIR)
    os.makedirs(out_subdir, exist_ok=True)

    # 从 folder1 或 folder2 复制 15 只股票的 CSV 到 15_common_stocks，文件名统一为 stock_代码.csv
    def _find_and_copy(code, from_path, out_dir):
        for f in os.listdir(from_path):
            if _stock_code_from_filename(f) == code:
                src = os.path.join(from_path, f)
                dst = os.path.join(out_dir, f"stock_{code}.csv")
                if os.path.normpath(src) != os.path.normpath(dst):
                    shutil.copy2(src, dst)
                return True
        return False

    for code in selected:
        if not _find_and_copy(code, path1, out_subdir):
            _find_and_copy(code, path2, out_subdir)

    if len(common) < NUM_COMMON:
        print(f"  共同股票数: {len(common)}（不足{NUM_COMMON}只）→ 已从两文件夹并集补足，共 {len(selected)} 只写入: {COMMON_STOCKS_DIR}/")
    else:
        print(f"  共同股票数: {len(common)}  →  已取前 {NUM_COMMON} 只写入: {COMMON_STOCKS_DIR}/")

print("\n" + "=" * 60)
print("统计汇总（可用于核对）")
print("=" * 60)
for name, subdirs, c1, c2 in stats_rows:
    print(f"  {name}: 子文件夹 {subdirs}, CSV 数 {c1} / {c2}")
print("\n完成。每个 *stocks_output 下均已生成 15_common_stocks 文件夹（含最多 15 只两文件夹共有/并集补足股票）。")

各 stocks_output 统计（每个目录下两个子文件夹及股票 CSV 数量）

【20160104stocks_output】
  子文件夹1: 15_common_stocks  →  股票 CSV 数: (不统计)
  子文件夹2: HTICST110.20160104.1  →  股票 CSV 数: (不统计)
  共同股票数: 15（不足300只）→ 已从两文件夹并集补足，共 300 只写入: 300_common_stocks/

【20160105stocks_output】
  子文件夹1: 15_common_stocks  →  股票 CSV 数: 15
  子文件夹2: HTICST110.20160105.1  →  股票 CSV 数: 3151
  共同股票数: 15（不足300只）→ 已从两文件夹并集补足，共 300 只写入: 300_common_stocks/

【20160106stocks_output】
  子文件夹1: 15_common_stocks  →  股票 CSV 数: 15
  子文件夹2: HTICST110.20160106.1  →  股票 CSV 数: 2961
  共同股票数: 15（不足300只）→ 已从两文件夹并集补足，共 300 只写入: 300_common_stocks/

【20160107stocks_output】
  子文件夹1: 15_common_stocks  →  股票 CSV 数: 15
  子文件夹2: HTICST110.20160107.1  →  股票 CSV 数: 2754
  共同股票数: 15（不足300只）→ 已从两文件夹并集补足，共 300 只写入: 300_common_stocks/

【20160108stocks_output】
  子文件夹1: 15_common_stocks  →  股票 CSV 数: 15
  子文件夹2: HTICST110.20160108.1  →  股票 CSV 数: 2503
  共同股票数: 15（不足300只）→ 已从两文件夹并集补足，共 300 只写入: 300_common_stocks/

【20160112stocks_output】
  子文件夹1: 15_common_stocks  →  股票 C

In [9]:
# ========== 将所有日期的 15_common_stocks 合并到同一文件夹（按日期分子目录） ==========
import os
import re
import shutil

BASE = os.getcwd()
COMMON_STOCKS_DIR = "15_common_stocks"
MERGED_DIR = "merged_15_common_stocks"

merged_root = os.path.join(BASE, MERGED_DIR)
os.makedirs(merged_root, exist_ok=True)

# 查找所有 *stocks_output（8位数字+stocks_output）
for name in sorted(os.listdir(BASE)):
    if not re.match(r"^\d{8}stocks_output$", name):
        continue
    date_token = name.replace("stocks_output", "")  # 如 20160104
    src_dir = os.path.join(BASE, name, COMMON_STOCKS_DIR)
    if not os.path.isdir(src_dir):
        continue
    dst_sub = os.path.join(merged_root, date_token)
    os.makedirs(dst_sub, exist_ok=True)
    count = 0
    for f in os.listdir(src_dir):
        if f.endswith(".csv"):
            src = os.path.join(src_dir, f)
            dst = os.path.join(dst_sub, f)
            if os.path.normpath(src) != os.path.normpath(dst):
                shutil.copy2(src, dst)
            count += 1
    print(f"  {name} → {MERGED_DIR}/{date_token}/  ({count} 个 CSV)")

print(f"\n完成。所有 15_common_stocks 已合并到: {merged_root}")

  20160104stocks_output → merged_300_common_stocks/20160104/  (300 个 CSV)
  20160105stocks_output → merged_300_common_stocks/20160105/  (300 个 CSV)
  20160106stocks_output → merged_300_common_stocks/20160106/  (300 个 CSV)
  20160107stocks_output → merged_300_common_stocks/20160107/  (300 个 CSV)
  20160108stocks_output → merged_300_common_stocks/20160108/  (300 个 CSV)
  20160112stocks_output → merged_300_common_stocks/20160112/  (300 个 CSV)
  20160113stocks_output → merged_300_common_stocks/20160113/  (300 个 CSV)
  20160114stocks_output → merged_300_common_stocks/20160114/  (300 个 CSV)
  20160115stocks_output → merged_300_common_stocks/20160115/  (300 个 CSV)
  20160118stocks_output → merged_300_common_stocks/20160118/  (300 个 CSV)
  20160119stocks_output → merged_300_common_stocks/20160119/  (300 个 CSV)
  20160120stocks_output → merged_300_common_stocks/20160120/  (300 个 CSV)
  20160121stocks_output → merged_300_common_stocks/20160121/  (300 个 CSV)
  20160122stocks_output → merged_300_c